# Random Dice Roll

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Simulation, Strings · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- **Input validation** — what counts as invalid, and how you signal it
- **Structural parsing** rather than character-by-character string poking
- API design: **where** errors are handled, and how you keep internal state safe

There is almost no algorithm here. It is a *code quality* question wearing a dice costume, and it is graded on the things production code is graded on: does it reject bad input clearly, does it use the language's conventions, and can a caller misuse it by accident?

**First-principles primer — what is each piece?**

- **`random.randint(a, b)`** — a uniformly random integer in `[a, b]`, **inclusive at both ends**. That inclusivity is why `randint(1, n)` is right and `randrange(1, n)` (which excludes `n`) is a classic off-by-one.
- **`TypeError` vs `ValueError`.** Python's convention: `TypeError` when the *kind* of thing is wrong (a string where an int was expected); `ValueError` when the type is right but the *value* is not (an int, but zero sides). Callers write `except ValueError:` to catch one and not the other, so getting this wrong makes their error handling unreliable.
- **Raising vs returning a sentinel.** A function should do **one** of these. Return `-1` on failure *and* raise on other failures, and every caller has to check both paths — and the one they forget is the one that bites.

**The three-line summary of the right shape:**

| Layer | Responsibility |
|---|---|
| `roll_die(n)` | validate `n`, roll once |
| `roll_multi("xDy")` | validate the *format*, then roll `x` times |
| `DiceRoller` | hold state (history); delegate the rolling |

Each layer validates only what it owns, and none of them swallow errors. The `try/except` the follow-up asks about belongs to the **caller** — a library that catches its own exceptions and returns `None` has just made every bug silent.

**The bug hiding in `isinstance(n, int)`.** In Python, `bool` is a **subclass of `int`**:

```python
isinstance(True, int)   # True  (!)
True == 1               # True
```

So a naive type check happily accepts `roll_die(True)` and rolls a one-sided die. If booleans should be rejected — and here they should, because `True` is not a meaningful number of sides — you must check for `bool` **explicitly and first**.

**Simple worked example.** `roll_multi("2D6")`:

```
"2D6".split("D")  ->  ["2", "6"]     exactly 2 parts  ✅
int("2") = 2, int("6") = 6           both parse       ✅
both >= 1                                             ✅
sum(randint(1,6) for _ in range(2))  ->  e.g. 4 + 3 = 7
```

And the malformed cases the split catches for free: `"2D"` → `["2", ""]` (empty part), `"D6"` → `["", "6"]`, `"2D6D4"` → 3 parts, `"abcD6"` → `int("abc")` raises.

## Problem Statement

**Problem 1** — `roll_die(n)`: return a uniformly random integer in `[1, n]`. Handle invalid input.

**Problem 2** — `roll_multi(spec)`: `spec` is `"xDy"` — roll `x` dice of `y` sides and return the **sum**. Reject input with no `"D"`, or with missing/malformed numbers.

**Problem 3** — maintain a **history** of inputs and their outputs.

```python
roll_die(6)          # -> 1..6
roll_multi("2D6")    # -> 2..12
roll_multi("3D7")    # -> 3..21

r = DiceRoller()
r.roll("2D6")        # -> 7
r.history()          # -> [("2D6", 7)]
```

### Problem 1 — a single die, with validation

**Idea:** validate first, roll second. Three rejections, each with a specific message:

1. **`bool`** — checked *before* `int`, because `bool` is a subclass of `int` and would otherwise slip through.
2. **not an `int`** — `TypeError`, per Python convention.
3. **`n < 1`** — `ValueError`: the type is fine, the value is not. A die needs at least one side.

Note what the function does **not** do: it does not catch its own exceptions. The follow-up's `try/except` belongs to whoever calls it.

**Time complexity:** O(1).

**Space complexity:** O(1).

In [ ]:
import random
from typing import List, Optional, Tuple


def roll_die(n: int) -> int:
    """Roll a single n-sided die. Returns an integer in [1, n] inclusive."""
    # bool is a SUBCLASS of int, so this check must come FIRST or True/False slip through.
    if isinstance(n, bool):
        raise TypeError("n must be an integer, got bool")
    if not isinstance(n, int):
        raise TypeError(f"n must be an integer, got {type(n).__name__}")
    if n < 1:
        raise ValueError(f"n must be at least 1, got {n}")
    return random.randint(1, n)          # randint is INCLUSIVE at both ends

### Problem 2 — Approach A: Naive (index arithmetic)

**Idea:** find the `"D"` with `str.find`, slice either side, and call `int()` on the pieces.

It works on well-formed input and fails confusingly on everything else — `"2D6D4"` silently parses as `2` and `"6D4"` (which then raises a `ValueError` from `int` with a message about `'6D4'`, not about the real problem). The failures are *late* and the messages point at the symptom rather than the cause.

**Time complexity:** O(x) for the rolls.

**Space complexity:** O(1).

In [ ]:
def roll_multi_naive(spec: str) -> int:
    i = spec.find("D")
    if i == -1:
        raise ValueError(f"spec must contain 'D', got {spec!r}")
    x = int(spec[:i])                    # raises a confusing error on "abcD6"
    y = int(spec[i + 1:])                # and silently mis-parses "2D6D4" as y="6D4"
    return sum(random.randint(1, y) for _ in range(x))

### Problem 2 — Approach B: Optimal (validate the structure, then parse)

**Idea:** check the **shape** of the input before trying to interpret any of it.

`split("D")` does most of the work in one call:

| input | split result | rejected because |
|---|---|---|
| `"2D6"` | `["2", "6"]` | ✅ valid |
| `"2D"` | `["2", ""]` | an empty part |
| `"D6"` | `["", "6"]` | an empty part |
| `"2D6D4"` | `["2", "6", "4"]` | not exactly 2 parts |
| `"26"` | `["26"]` | not exactly 2 parts (no `"D"`) |
| `"abcD6"` | `["abc", "6"]` | `int("abc")` fails |

Then parse, then range-check. Each stage produces an error message about **that** stage, so a caller reading the exception learns what was actually wrong.

`sum(... for ...)` uses a **generator**, not a list comprehension, so rolling 10 million dice does not allocate 10 million integers.

**Time complexity:** **O(x)** — one `randint` per die.

**Space complexity:** **O(1)** — the generator streams.

In [ ]:
def roll_multi(spec: str) -> int:
    """Parse an 'xDy' string and return the sum of x rolls of a y-sided die."""
    if not isinstance(spec, str):
        raise TypeError(f"spec must be a string, got {type(spec).__name__}")

    parts = spec.upper().split("D")      # accept "2d6" as well as "2D6"
    if len(parts) != 2:                  # catches no D, and more than one D
        raise ValueError(f"spec must be of the form 'xDy', got {spec!r}")

    x_str, y_str = (p.strip() for p in parts)
    if not x_str or not y_str:           # catches "2D" and "D6"
        raise ValueError(f"spec must have numbers on both sides of 'D', got {spec!r}")

    try:
        x, y = int(x_str), int(y_str)
    except ValueError:
        raise ValueError(f"x and y must be valid integers, got {spec!r}") from None

    if x < 1:
        raise ValueError(f"x must be at least 1, got {x}")
    if y < 1:
        raise ValueError(f"y must be at least 1, got {y}")

    # A GENERATOR, not a list comprehension: rolling 10M dice allocates nothing.
    return sum(random.randint(1, y) for _ in range(x))

### Problem 3 — history, with a bounded option

**Idea:** state that survives across calls means a class. Two design decisions carry the marks:

- **`history()` returns a copy.** Handing back `self._history` lets a caller do `r.history().clear()` and silently wipe your internal state. A shallow copy is one word and closes that hole.
- **A bounded history uses `deque(maxlen=...)`.** The obvious `list.append` + `list.pop(0)` is **O(n) per eviction**, because popping the front shifts every remaining element. A `deque` evicts in O(1) and needs no length check at all — `maxlen` does it. Unbounded history in a long-running process is a real memory leak, so offering the cap unprompted is worth saying.

Note the roll is only recorded **after** `roll_multi` returns: a rejected spec must not pollute the history.

**Time complexity:** O(x) per roll; O(1) to record.

**Space complexity:** O(h) for the retained history.

In [ ]:
from collections import deque


class DiceRoller:
    """Rolls dice and remembers (spec, result) pairs."""

    def __init__(self, max_history: Optional[int] = None) -> None:
        # deque(maxlen=...) evicts the oldest in O(1); list.pop(0) would be O(n).
        self._history = deque(maxlen=max_history) if max_history else deque()
        self.max_history = max_history

    def roll(self, spec: str) -> int:
        result = roll_multi(spec)            # may raise - recorded ONLY on success
        self._history.append((spec, result))
        return result

    def history(self) -> List[Tuple[str, int]]:
        return list(self._history)           # a COPY: callers cannot mutate our state

    def clear_history(self) -> None:
        self._history.clear()

    def __len__(self) -> int:
        return len(self._history)

### Follow-up — modifiers and keep-highest (`"2D6+3"`, `"2D20k1"`)

**Idea:** the format generalises to `xDy[k<n>][+/-m]`, which is standard tabletop dice notation:

- **`+m` / `-m`** — a flat modifier added to the total.
- **`kN`** — roll `x` dice but **keep the highest N**. This is how advantage works in D&D 5e: `2D20k1` is "roll twice, take the better".

The parsing stays structural — peel the modifier off the end, then the keep clause, then hand the remaining `xDy` to the existing validator. Reusing `roll_multi`'s validation rather than re-implementing it is the point.

One genuine change: `kN` needs the **individual rolls**, not just their sum, so the roll step must produce a list before reducing it. That is why the helper returns the list and the caller decides how to combine.

**Time complexity:** O(x log x) when a keep clause is present (the sort), O(x) otherwise.

**Space complexity:** O(x) — the individual rolls must be materialised to select from them.

In [ ]:
import re

SPEC_RE = re.compile(r"^\s*(\d+)\s*[dD]\s*(\d+)\s*(?:[kK]\s*(\d+)\s*)?(?:([+-])\s*(\d+)\s*)?$")


def roll_advanced(spec: str) -> int:
    """Supports xDy, xDy+m, xDy-m, xDykN (keep the highest N), and combinations."""
    if not isinstance(spec, str):
        raise TypeError(f"spec must be a string, got {type(spec).__name__}")
    m = SPEC_RE.match(spec)
    if not m:
        raise ValueError(f"spec must be of the form 'xDy[kN][+/-m]', got {spec!r}")

    x, y = int(m.group(1)), int(m.group(2))
    keep = int(m.group(3)) if m.group(3) else None
    sign, mod = m.group(4), int(m.group(5)) if m.group(5) else 0

    if x < 1:
        raise ValueError(f"x must be at least 1, got {x}")
    if y < 1:
        raise ValueError(f"y must be at least 1, got {y}")
    if keep is not None and not (1 <= keep <= x):
        raise ValueError(f"keep must be between 1 and {x}, got {keep}")

    rolls = [random.randint(1, y) for _ in range(x)]      # a LIST: keep-N must select from them
    if keep is not None:
        rolls = sorted(rolls, reverse=True)[:keep]        # advantage = keep the highest
    total = sum(rolls)
    return total + mod if sign == "+" else total - mod if sign == "-" else total

## Verification

Randomness makes this awkward to test, so the assertions below check the things that are **deterministic**: bounds, distribution coverage, and — above all — that every malformed input is rejected with the right exception type.

In [ ]:
import random as _random

random.seed(53)

# --- Problem 1: bounds and coverage ---
for n in (1, 2, 6, 20, 100):
    seen = set()
    for _ in range(4000):
        v = roll_die(n)
        assert 1 <= v <= n, f"roll_die({n}) returned {v}, outside [1, {n}]"
        seen.add(v)
    if n <= 20:
        assert seen == set(range(1, n + 1)), (
            f"roll_die({n}) never produced {set(range(1, n+1)) - seen} - "
            "randint must be INCLUSIVE at both ends"
        )
assert roll_die(1) == 1, "a 1-sided die can only ever return 1"

# --- Problem 1: invalid input ---
for bad in (0, -1, -100):
    try:
        roll_die(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_die({bad}) must raise ValueError")

for bad in ("6", 6.0, None, [6], {"n": 6}):
    try:
        roll_die(bad)
    except TypeError:
        pass
    else:
        raise AssertionError(f"roll_die({bad!r}) must raise TypeError")

# THE bool trap: bool is a subclass of int, so a naive isinstance check lets it through
for b in (True, False):
    try:
        roll_die(b)
    except TypeError:
        pass
    else:
        raise AssertionError(f"roll_die({b}) must raise TypeError - bool is not a side count")
assert isinstance(True, int), "this is WHY the explicit bool check is needed"

# --- Problem 2: bounds ---
for spec, lo, hi in [("1D6", 1, 6), ("2D6", 2, 12), ("3D7", 3, 21), ("1D1", 1, 1),
                     ("10D2", 10, 20), ("2d6", 2, 12)]:
    seen = set()
    for _ in range(3000):
        v = roll_multi(spec)
        assert lo <= v <= hi, f"{spec} returned {v}, outside [{lo}, {hi}]"
        seen.add(v)
    assert lo in seen and hi in seen, f"{spec} never hit both extremes {lo} and {hi}"

assert roll_multi("1D1") == 1
assert roll_multi("5D1") == 5, "five 1-sided dice always sum to 5"
assert roll_multi(" 2 D 6 ") in range(2, 13), "surrounding whitespace is tolerated"

# --- Problem 2: every documented invalid case ---
for bad in ["26", "2X6", "", "abc"]:                      # no 'D'
    try:
        roll_multi(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise ValueError (no 'D')")

for bad in ["2D", "D6", "D", "  D  "]:                    # a missing number
    try:
        roll_multi(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise ValueError (missing number)")

for bad in ["abcD6", "2Dxyz", "2.5D6", "2D6.5"]:          # not integers
    try:
        roll_multi(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise ValueError (not an integer)")

for bad in ["2D6D4", "1D2D3D4"]:                          # more than one 'D'
    try:
        roll_multi(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise ValueError (multiple 'D')")

for bad in ["0D6", "2D0", "-1D6", "2D-6"]:                # out of range
    try:
        roll_multi(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise ValueError (non-positive)")

for bad in (26, None, ["2D6"]):                           # not a string
    try:
        roll_multi(bad)
    except TypeError:
        pass
    else:
        raise AssertionError(f"roll_multi({bad!r}) must raise TypeError")

# The naive parser mis-handles a case the structural one catches
try:
    roll_multi_naive("2D6D4")
except ValueError as e:
    assert "6D4" in str(e), "the naive parser's error blames the wrong thing"

# --- Problem 3: history ---
r = DiceRoller()
assert r.history() == [] and len(r) == 0
v1 = r.roll("2D6")
v2 = r.roll("3D7")
h = r.history()
assert h == [("2D6", v1), ("3D7", v2)], h
assert 2 <= v1 <= 12 and 3 <= v2 <= 21

# A rejected spec must NOT be recorded
try:
    r.roll("bad")
except ValueError:
    pass
assert len(r.history()) == 2, "a failed roll must not enter the history"

# history() returns a COPY - mutating it must not corrupt internal state
snapshot = r.history()
snapshot.clear()
snapshot.append(("fake", 999))
assert len(r.history()) == 2, "history() must hand back a copy, not the live list"

r.clear_history()
assert r.history() == [] and len(r) == 0

# Bounded history evicts the oldest
capped = DiceRoller(max_history=3)
for i in range(10):
    capped.roll("1D6")
assert len(capped.history()) == 3, "max_history must cap the stored entries"

# --- Follow-up: modifiers and keep-highest ---
for _ in range(2000):
    assert 5 <= roll_advanced("2D6+3") <= 15                # 2..12, plus 3
    assert -1 <= roll_advanced("2D6-3") <= 9                # 2..12, minus 3
    assert 1 <= roll_advanced("2D20k1") <= 20               # keep the best of two d20s
    assert 2 <= roll_advanced("3D6k2") <= 12                # keep the best two of three d6s
    assert 2 <= roll_advanced("2D6") <= 12                  # plain form still works

assert roll_advanced("3D1k2") == 2, "keep 2 of three 1-sided dice -> 1 + 1"
assert roll_advanced("2D1+5") == 7
assert roll_advanced("2D1-1") == 1

# Keeping the highest must beat the plain average over many trials
plain = sum(roll_advanced("2D20") for _ in range(4000)) / 4000
adv = sum(roll_advanced("2D20k1") for _ in range(4000)) / 4000
assert adv < plain, "keeping ONE of two dice sums less than keeping both"
best_of_two = sum(roll_advanced("2D20k1") for _ in range(4000)) / 4000
single = sum(roll_die(20) for _ in range(4000)) / 4000
assert best_of_two > single, "advantage (best of 2) must beat a single d20 on average"

for bad in ["2D20k0", "2D20k3", "2D6+", "2D6++3", "xD6k1"]:
    try:
        roll_advanced(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"roll_advanced({bad!r}) must raise ValueError")

# --- The distribution is roughly uniform (a weak but real sanity check) ---
counts = [0] * 7
for _ in range(60000):
    counts[roll_die(6)] += 1
expected = 60000 / 6
for face in range(1, 7):
    assert abs(counts[face] - expected) < expected * 0.1, (
        f"face {face} appeared {counts[face]} times, expected about {expected:.0f}"
    )

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Where the `try/except` belongs.** The follow-up asks for error handling, and the tempting reading is "wrap the body in `try/except` and return `None` on failure". That is the wrong lesson. A library that swallows its own exceptions turns every caller bug into a silent wrong answer — `roll_multi("2D")` returning `None` propagates a `None` into someone's arithmetic and fails a hundred lines later. **Raise at the point of the mistake; catch at the boundary where you can actually do something** (retry, prompt the user, log and skip).
- **Thread safety.** `random`'s module-level functions share one global generator instance; CPython's GIL makes each individual call atomic, so you will not get corrupted output, but you *will* get interleaved sequences that are not reproducible from a seed. For reproducibility per thread, give each one its own `random.Random(seed)` instance. `DiceRoller.roll` also needs a lock if you care about history ordering — `deque.append` is itself thread-safe, but "roll then append" is two steps and another thread can interleave between them.
- **`roll_many([...])` — validate first, or roll as you go?** A real fork. **Validate all, then roll** is atomic: either every spec is good and you get every result, or nothing happened. **Roll as you go** gives partial results and fails halfway, leaving the caller with an inconsistent set. Prefer all-or-nothing unless the rolls have side effects that cannot be undone — and note that validating first requires a `validate(spec)` that does not roll, which means factoring the parser apart from the roller.
- **Persisting the history.** JSON Lines (one `{"spec": ..., "result": ...}` per line) is append-only, human-readable, and survives a crash mid-write with at most one corrupt trailing line — see [`3. Persistent_Append_Only_Log`](../3.%20Persistent_Append_Only_Log/3.%20Persistent_Append_Only_Log.ipynb) for why that shape is so common. SQLite is the answer once you want to *query* the history ("all rolls of 2D6 last week"). Either way, decide whether a roll is acknowledged before or after the write is durable.
- **Testing randomness properly.** The assertions above check bounds, that both extremes are reachable (which catches inclusive/exclusive off-by-ones), and rough uniformity. For anything stronger, **seed the generator** or inject it as a dependency so the sequence is reproducible. A chi-squared test is the principled version of the uniformity check — worth naming, though for an interview the bounds-plus-coverage checks are what actually catch bugs.

## Empirical complexity check

`roll_multi` is O(x) — one random draw per die — so the runtime should track the die count exactly. The comparison worth making is the **generator vs list** choice inside `sum`: identical time, but the list version allocates `x` integers it never needs.

| Growth when the die count doubles | What it means |
|---|---|
| ~2x | linear — one `randint` per die, as expected |

Both rows should sit at ~2x; the point of running both is that the *time* is the same while only one of them holds the whole roll set in memory at once.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def make_spec(x):
    return (f"{x}D6",)


def run_generator(spec):
    roll_multi(spec)                     # sum(generator): O(1) space


def run_list(spec):
    x, y = spec.upper().split("D")
    sum([random.randint(1, int(y)) for _ in range(int(x))])   # O(x) space, same time


benchmark(
    {"sum(generator) - O(1) space": run_generator,
     "sum([list])    - O(x) space": run_list},
    make_spec,
    sizes=[20000, 40000, 80000, 160000],
    repeats=2,
)

## Patterns learned

- **Validate at the boundary, raise at the mistake, catch where you can act.** A function that swallows its own errors converts loud bugs into silent wrong answers. The `try/except` belongs to the caller.
- **`TypeError` for the wrong kind, `ValueError` for the wrong value.** Callers write `except` clauses against these; using the wrong one makes their error handling unreliable.
- **`bool` is a subclass of `int`.** `isinstance(True, int)` is `True`. Any type check that must exclude booleans has to say so explicitly, and first.
- **Check the shape before parsing the pieces.** `split` + a length check rejects missing numbers, extra separators and empty strings in one pass, with an error message about the actual problem — far better than index arithmetic that fails later and blames the wrong thing.
- **Never hand out your internal collection.** `return list(self._history)`. One word, and a caller can no longer corrupt your state by accident.
- **Bound anything that grows per operation.** `deque(maxlen=n)` caps memory *and* evicts in O(1); `list.pop(0)` is O(n) and a quiet quadratic term.
- **`sum(generator)` not `sum([list])`.** Same time, O(1) space instead of O(n). Free, once you notice.
- **Test randomness by its invariants.** Bounds always hold; both extremes must be reachable (which catches inclusive/exclusive off-by-ones); the distribution should be roughly flat. Seed the generator when you need reproducibility.